# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

Question: *which visible pages under-capture clicks for the position they already hold, and deserve a title / meta / snippet review first?*

Why this lane:
- **It sits on demand that already exists.** A page on page 1 with thousands of impressions is already in front of searchers; the gap is the click. That is a cheaper fix than earning new rankings.
- **The comparison is fair and explainable.** CTR depends heavily on position, so I compare every page only to pages in the same position tier. The baseline is a sentence a content editor can check: *"this page gets half the clicks its position tier normally gets."*
- **It matches what I do at work.** I'm a data analyst; benchmarking an item against its own peer group and turning the gap into a ranked list is the kind of analysis I can defend.

I can still change lanes until the end of Week 4. If the signal turns out to be mostly noise or position artefacts, Lane 2 (refresh opportunity scoring) is my fallback.

In [1]:
# Lane choice, recorded as data so later notebooks can reuse it
LANE = "Lane 4: CTR / Engagement Opportunity Scoring"
UNIT_OF_ANALYSIS = "one content item (page), trailing 90-day window"
print(LANE)
print("Unit of analysis:", UNIT_OF_ANALYSIS)

Lane 4: CTR / Engagement Opportunity Scoring
Unit of analysis: one content item (page), trailing 90-day window


## 2. The question: decision, action, cost of a wrong call

| | |
|---|---|
| **Search question** | Which visible pages get clearly fewer clicks than pages at a similar position, with enough impressions that the gap is not noise? |
| **Unit of analysis** | One page (content item), using its trailing 90-day metrics. |
| **Output** | A ranked review queue: each page with its CTR gap vs. its position tier, the clicks it is missing at that tier's CTR, and reason codes (e.g. *high impressions*, *page-1 position*, *CTR below half of tier*). |
| **Decision it improves** | Which pages a content editor should review first, given limited hours. |
| **Who acts, and how** | A content editor / SEO specialist opens the top of the queue and rewrites title and meta description, checks intent match and snippet structure, or marks the page *monitor* if the gap has an outside explanation (an AI Overview or other SERP feature above it, a navigational query). |
| **Cost of a wrong call** | *False positive:* an editor spends ~30–60 minutes rewriting a page whose low CTR is caused by the results page itself; the edit can't help and might even hurt a working title. *False negative:* a high-impression page keeps leaking clicks for another quarter. With fixed editor capacity, false positives at the top of the queue are the more expensive error, so I will judge the queue by **precision@20 / @50**, not accuracy. |

**Why data/ML helps at all (and why this is not just "train a model"):**
- A raw CTR sort is misleading, because position drives CTR. The first honest step is a **rule**: compare each page to its own position tier's expected CTR. That rule is my baseline, and the model has to beat it.
- ML earns a place only if the gap depends on several tangled factors at once — intent, content type, impression volume, freshness, engagement — that a single rule can't capture well. I will test that, not assume it.
- The target has to be **observed**, not defined by a product rule. The likely candidate: whether CTR stays below its tier in a later window, built from the warehouse daily data. That stops me from learning one noisy 90-day snapshot.

In [2]:
# One-paragraph frame (the framing-ml-problems template), kept next to the code
FRAME = (
    "For content editors deciding which pages to review first, we will build a ranked review "
    "queue from the FlyRank starter/warehouse data, scoring each visible page by its CTR gap "
    "versus pages in the same position tier, measured by precision@20/@50 on a later window. "
    "A wrong call costs editor hours on pages a title rewrite cannot fix. A plain CTR sort is not "
    "enough because position drives CTR; the tier-adjusted rule is the baseline any model must beat. "
    "We will claim only observed, directional, decision-support results."
)
print(FRAME)

For content editors deciding which pages to review first, we will build a ranked review queue from the FlyRank starter/warehouse data, scoring each visible page by its CTR gap versus pages in the same position tier, measured by precision@20/@50 on a later window. A wrong call costs editor hours on pages a title rewrite cannot fix. A plain CTR sort is not enough because position drives CTR; the tier-adjusted rule is the baseline any model must beat. We will claim only observed, directional, decision-support results.


## 3. Quick look at the data (2-3 real numbers)

Below: aggregates only (no row-level output, no IDs). Rates are on the ×100 scale from the data dictionary: `0.35` means 0.35%. Pages with `avg_position = 0` (no position data) fall into the `no_data` tier and are left out of the tier comparison.

In [3]:
import pandas as pd
from pathlib import Path

# find the repo root whether the notebook runs from work/notebooks/ or the repo root
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,} pages | clients: {df['client_id'].nunique()}")

# Number 1: CTR depends heavily on position tier -> raw CTR comparisons across tiers are unfair
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
by_tier = (df[df["position_tier"].isin(tier_order)]
           .groupby("position_tier")[["impressions_90d", "clicks_90d"]].sum()
           .reindex(tier_order))
by_tier["pages"] = df["position_tier"].value_counts().reindex(tier_order)
by_tier["pooled_ctr_%"] = (100 * by_tier["clicks_90d"] / by_tier["impressions_90d"]).round(3)
print("\nNumber 1: pooled CTR by position tier")
print(by_tier[["pages", "pooled_ctr_%"]])

# Number 2: among visible pages with enough volume, how many sit far below their own tier?
MIN_IMPRESSIONS = 1_000           # policy choice: below this, CTR is too noisy to act on
visible = df[df["position_tier"].isin(["top_3", "page_1", "striking"]) & (df["impressions_90d"] >= MIN_IMPRESSIONS)].copy()
visible["tier_ctr"] = visible["position_tier"].map(by_tier["pooled_ctr_%"])
visible["below_half"] = visible["ctr"] < 0.5 * visible["tier_ctr"]
share_pages = visible["below_half"].mean()
share_impr = visible.loc[visible["below_half"], "impressions_90d"].sum() / visible["impressions_90d"].sum()
print(f"\nNumber 2: visible pages (positions 1-20, >= {MIN_IMPRESSIONS:,} impressions): {len(visible):,}")
print(f"  with CTR below half of their tier's pooled CTR: {share_pages:.1%} of pages, holding {share_impr:.1%} of these pages' impressions")

# Number 3: spread of CTR inside one tier (page 1, positions 4-10)
p1 = visible.loc[visible["position_tier"] == "page_1", "ctr"]
q1, q2, q3 = p1.quantile([0.25, 0.5, 0.75])
print(f"\nNumber 3: page-1 CTR quartiles: {q1:.2f}% / {q2:.2f}% / {q3:.2f}%  (75th / 25th = {q3/q1:.1f}x spread at a similar position)")

# Directional size of the prize (an upper bound, not a forecast)
missing = (visible["tier_ctr"] / 100 * visible["impressions_90d"] - visible["clicks_90d"]).clip(lower=0)
print(f"\nDirectional only: clicks missing vs. tier average on the below-half pages: "
      f"{missing[visible['below_half']].sum():,.0f} over 90 days (all visible pages earned {visible['clicks_90d'].sum():,.0f})")

Rows: 30,000 pages | clients: 32

Number 1: pooled CTR by position tier
               pages  pooled_ctr_%
position_tier                     
top_3           2321         0.488
page_1         11814         0.350
striking        7304         0.347
page_3_5        7242         0.155
deep            1319         0.041

Number 2: visible pages (positions 1-20, >= 1,000 impressions): 9,998
  with CTR below half of their tier's pooled CTR: 40.5% of pages, holding 33.0% of these pages' impressions

Number 3: page-1 CTR quartiles: 0.13% / 0.24% / 0.45%  (75th / 25th = 3.5x spread at a similar position)

Directional only: clicks missing vs. tier average on the below-half pages: 99,993 over 90 days (all visible pages earned 420,768)


## 4. Careful words: what I can and can't claim

**What this work will be able to say**
- *Observed:* which pages had a lower CTR than their position tier over the measured window.
- *Measured:* how big the gap is, in percentage points and in clicks at tier-average CTR.
- *Directional:* whether pages flagged by the rule or model tend to stay below their tier in a later window (precision@K on held-out clients).
- *Decision-support:* a ranked list of pages **worth a human review first**, each with reason codes an editor can check.

**What it will never say**
- That a title or meta rewrite **will** raise CTR. This data is observational; proving the effect of an edit needs an experiment.
- Anything about how Google's algorithm ranks or displays pages, or about AI Overviews, AI citations or AI rankings. Low CTR can come from the results page itself, which this data does not show.
- That the "missing clicks" number is lost revenue or a forecast. It is an upper-bound illustration at tier-average CTR.

**Known traps I will handle**
- Position drives CTR, so I compare only within a position tier, and finer position buckets later. One warning sign already shows up: `striking` (positions 11–20) has almost the same pooled CTR as `page_1` (0.347% vs 0.350%). A 90-day *average* position probably blends pages that moved between tiers, so tier boundaries need checking before I trust them.
- Low-volume pages are noisy, so I use a minimum-impressions threshold and record it as a policy choice.
- `avg_position = 0` means no data, not rank zero.
- `trend_direction` / `trend_pct` are never features, and `client_id` / `content_id` are used only for grouped splits.
- No row-level dumps, IDs, URLs or queries in any public output.

In [4]:
# Guardrail check: the columns this lane will start from, and the ones it must never use as features
SAFE_START = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
              "main_intent", "content_type", "days_since_last_update", "engagement_rate", "sessions_90d"]
NEVER_FEATURES = ["trend_direction", "trend_pct", "content_id", "client_id"]
assert all(c in df.columns for c in SAFE_START + NEVER_FEATURES)
print("Start features:", ", ".join(SAFE_START))
print("Never features:", ", ".join(NEVER_FEATURES))

Start features: impressions_90d, clicks_90d, ctr, avg_position, position_tier, main_intent, content_type, days_since_last_update, engagement_rate, sessions_90d
Never features: trend_direction, trend_pct, content_id, client_id


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.